# TP DES & AES — Partie 3 : implémentation de DES

Implémentation de DES (FIPS 46-3) sans bibliothèque, en Python. Les blocs sont manipulés comme des **entiers** (64 bits) et les tables sont reprises de la norme (les positions sont numérotées de 1 à n, bit 1 = bit de poids fort).

Structure générale :
- **Clé** 64 bits (dont 8 bits de parité ignorés → 56 bits effectifs) → PC-1 → 16 tours de décalages circulaires + PC-2 → 16 sous-clés de 48 bits
- **Bloc** 64 bits → IP → 16 tours de Feistel → échange final des moitiés → FP = IP⁻¹

## Tables de DES

In [1]:
# Permutation initiale
IP = [58,50,42,34,26,18,10,2, 60,52,44,36,28,20,12,4,
      62,54,46,38,30,22,14,6, 64,56,48,40,32,24,16,8,
      57,49,41,33,25,17,9,1,  59,51,43,35,27,19,11,3,
      61,53,45,37,29,21,13,5, 63,55,47,39,31,23,15,7]

# Permutation finale (IP^-1)
FP = [40,8,48,16,56,24,64,32, 39,7,47,15,55,23,63,31,
      38,6,46,14,54,22,62,30, 37,5,45,13,53,21,61,29,
      36,4,44,12,52,20,60,28, 35,3,43,11,51,19,59,27,
      34,2,42,10,50,18,58,26, 33,1,41,9,49,17,57,25]

# Expansion 32 -> 48 bits
E = [32,1,2,3,4,5,     4,5,6,7,8,9,     8,9,10,11,12,13,   12,13,14,15,16,17,
     16,17,18,19,20,21, 20,21,22,23,24,25, 24,25,26,27,28,29, 28,29,30,31,32,1]

# Permutation P (P-box) après les S-box
P = [16,7,20,21,29,12,28,17, 1,15,23,26,5,18,31,10,
     2,8,24,14,32,27,3,9,    19,13,30,6,22,11,4,25]

# Permuted Choice 1 : 64 -> 56 bits (suppression des bits de parité)
PC1 = [57,49,41,33,25,17,9,  1,58,50,42,34,26,18,
       10,2,59,51,43,35,27,  19,11,3,60,52,44,36,
       63,55,47,39,31,23,15, 7,62,54,46,38,30,22,
       14,6,61,53,45,37,29,  21,13,5,28,20,12,4]

# Permuted Choice 2 (permutation de contraction) : 56 -> 48 bits
PC2 = [14,17,11,24,1,5,   3,28,15,6,21,10,
       23,19,12,4,26,8,   16,7,27,20,13,2,
       41,52,31,37,47,55, 30,40,51,45,33,48,
       44,49,39,56,34,53, 46,42,50,36,29,32]

# Décalages circulaires à gauche par tour
SHIFTS = [1,1,2,2,2,2,2,2,1,2,2,2,2,2,2,1]

# S-boxes : S[i][ligne][colonne]
S = [
 [[14,4,13,1,2,15,11,8,3,10,6,12,5,9,0,7],[0,15,7,4,14,2,13,1,10,6,12,11,9,5,3,8],
  [4,1,14,8,13,6,2,11,15,12,9,7,3,10,5,0],[15,12,8,2,4,9,1,7,5,11,3,14,10,0,6,13]],
 [[15,1,8,14,6,11,3,4,9,7,2,13,12,0,5,10],[3,13,4,7,15,2,8,14,12,0,1,10,6,9,11,5],
  [0,14,7,11,10,4,13,1,5,8,12,6,9,3,2,15],[13,8,10,1,3,15,4,2,11,6,7,12,0,5,14,9]],
 [[10,0,9,14,6,3,15,5,1,13,12,7,11,4,2,8],[13,7,0,9,3,4,6,10,2,8,5,14,12,11,15,1],
  [13,6,4,9,8,15,3,0,11,1,2,12,5,10,14,7],[1,10,13,0,6,9,8,7,4,15,14,3,11,5,2,12]],
 [[7,13,14,3,0,6,9,10,1,2,8,5,11,12,4,15],[13,8,11,5,6,15,0,3,4,7,2,12,1,10,14,9],
  [10,6,9,0,12,11,7,13,15,1,3,14,5,2,8,4],[3,15,0,6,10,1,13,8,9,4,5,11,12,7,2,14]],
 [[2,12,4,1,7,10,11,6,8,5,3,15,13,0,14,9],[14,11,2,12,4,7,13,1,5,0,15,10,3,9,8,6],
  [4,2,1,11,10,13,7,8,15,9,12,5,6,3,0,14],[11,8,12,7,1,14,2,13,6,15,0,9,10,4,5,3]],
 [[12,1,10,15,9,2,6,8,0,13,3,4,14,7,5,11],[10,15,4,2,7,12,9,5,6,1,13,14,0,11,3,8],
  [9,14,15,5,2,8,12,3,7,0,4,10,1,13,11,6],[4,3,2,12,9,5,15,10,11,14,1,7,6,0,8,13]],
 [[4,11,2,14,15,0,8,13,3,12,9,7,5,10,6,1],[13,0,11,7,4,9,1,10,14,3,5,12,2,15,8,6],
  [1,4,11,13,12,3,7,14,10,15,6,8,0,5,9,2],[6,11,13,8,1,4,10,7,9,5,0,15,14,2,3,12]],
 [[13,2,8,4,6,15,11,1,10,9,3,14,5,0,12,7],[1,15,13,8,10,3,7,4,12,5,6,11,0,14,9,2],
  [7,11,4,1,9,12,14,2,0,6,10,13,15,3,5,8],[2,1,14,7,4,10,8,13,15,12,9,0,3,5,6,11]],
]

## Permutation générique

Toutes les permutations de DES (IP, FP, E, P, PC-1, PC-2) ont la même forme : le bit $j$ de la sortie est le bit `table[j]` de l'entrée. Une seule fonction suffit. Pour E et PC-2, la table a une taille différente de l'entrée (expansion 32→48, contraction 56→48).

In [2]:
def permute(x, table, n_in):
    """x : entier de n_in bits. Renvoie l'entier de len(table) bits permuté."""
    out = 0
    for pos in table:
        out = (out << 1) | ((x >> (n_in - pos)) & 1)
    return out

## Cadencement de clé (key schedule)

1. **PC-1** : 64 → 56 bits (on retire les bits de parité 8, 16, …, 64) puis on coupe en deux moitiés $C_0$, $D_0$ de 28 bits.
2. Pour chaque tour $i$ : **décalage circulaire à gauche** de $C$ et $D$ de 1 ou 2 positions (table `SHIFTS`).
3. **PC-2** (permutation de contraction) : on sélectionne 48 des 56 bits de $C_i D_i$ → sous-clé $K_i$.

In [3]:
def rotl28(x, s):
    return ((x << s) | (x >> (28 - s))) & 0xFFFFFFF

def key_schedule(key):
    """key : entier 64 bits -> liste des 16 sous-clés de 48 bits"""
    k = permute(key, PC1, 64)
    C, D = k >> 28, k & 0xFFFFFFF
    subkeys = []
    for s in SHIFTS:
        C, D = rotl28(C, s), rotl28(D, s)
        subkeys.append(permute((C << 28) | D, PC2, 56))
    return subkeys

## Fonction de tour $f(R, K)$

1. **Expansion E** : $R$ (32 bits) → 48 bits (certains bits sont dupliqués).
2. **XOR** avec la sous-clé $K_i$ (48 bits).
3. **Substitution S-box** : on coupe en 8 groupes de 6 bits $b_1..b_6$. Pour la S-box $i$ : ligne = $b_1b_6$, colonne = $b_2b_3b_4b_5$ → sortie de 4 bits. Total 32 bits. C'est la seule étape **non linéaire** de DES (confusion).
4. **Permutation P** : mélange les 32 bits pour que les sorties d'une S-box alimentent plusieurs S-box au tour suivant (diffusion).

Version « lisible » (directement d'après la norme) :

In [4]:
def sbox_sub(x48):
    out = 0
    for i in range(8):
        b = (x48 >> (42 - 6 * i)) & 0x3F          # 6 bits pour la S-box i
        row = ((b >> 4) & 0b10) | (b & 1)         # bits 1 et 6
        col = (b >> 1) & 0xF                      # bits 2 à 5
        out = (out << 4) | S[i][row][col]
    return out

def f_simple(R, K):
    return permute(sbox_sub(permute(R, E, 32) ^ K), P, 32)

def des_block_simple(block, subkeys):
    b = permute(block, IP, 64)                    # permutation initiale
    L, R = b >> 32, b & 0xFFFFFFFF
    for K in subkeys:                             # 16 tours de Feistel
        L, R = R, L ^ f_simple(R, K)
    return permute((R << 32) | L, FP, 64)         # échange final + permutation finale

## Optimisation : tables de permutation pré-calculées

La fonction `permute` fait une boucle bit par bit (64 itérations pour IP, 48 pour E…). Comme ça on obtient environ 9 000 blocs/s, ce qui est trop lent pour la partie 4 (2²⁶ octets = 8 millions de blocs).

Astuce classique : une permutation est **linéaire vis-à-vis du OU**, donc on peut traiter l'entrée **octet par octet**. Pour chaque position d'octet $j$ et chaque valeur $v \in [0,255]$, on pré-calcule `permute(v << position_j)`. La permutation devient alors un OU de 8 (ou 4) lectures de table au lieu de 64 décalages. On fait pareil pour les S-box (une table de 64 entrées directement indexée par les 6 bits). Les tables sont construites **avec la fonction `permute` ci-dessus**, donc le résultat est strictement le même.

In [5]:
def make_tables(table, n_in):
    nb = n_in // 8
    return [[permute(v << (8 * (nb - 1 - j)), table, n_in) for v in range(256)] for j in range(nb)]

T_IP = make_tables(IP, 64)
T_FP = make_tables(FP, 64)
T_E  = make_tables(E, 32)
T_P  = make_tables(P, 32)
# S-box "à plat" : S_FLAT[i][b] pour b sur 6 bits
S_FLAT = [[S[i][((b >> 4) & 2) | (b & 1)][(b >> 1) & 0xF] for b in range(64)] for i in range(8)]

def fast_perm(x, T):
    out, nb = 0, len(T)
    for j in range(nb):
        out |= T[j][(x >> (8 * (nb - 1 - j))) & 0xFF]
    return out

def f(R, K):
    x = fast_perm(R, T_E) ^ K
    out = 0
    for i in range(8):
        out = (out << 4) | S_FLAT[i][(x >> (42 - 6 * i)) & 0x3F]
    return fast_perm(out, T_P)

def des_block(block, subkeys):
    b = fast_perm(block, T_IP)
    L, R = b >> 32, b & 0xFFFFFFFF
    for K in subkeys:
        L, R = R, L ^ f(R, K)
    return fast_perm((R << 32) | L, T_FP)

## Chiffrement / déchiffrement de messages (mode ECB)

- Déchiffrer = même algorithme avec les sous-clés **dans l'ordre inverse** (propriété du réseau de Feistel).
- `pad=True` : padding PKCS#7 pour les messages de taille quelconque. Pour les mesures de performance (tailles multiples de 8) on utilise `pad=False`, comme pour la bibliothèque.

In [6]:
def des_encrypt(data, key, pad=True):
    """data, key : bytes (clé de 8 octets)"""
    sk = key_schedule(int.from_bytes(key, 'big'))
    if pad:
        n = 8 - len(data) % 8
        data = data + bytes([n]) * n
    out = bytearray(len(data))
    for i in range(0, len(data), 8):
        c = des_block(int.from_bytes(data[i:i+8], 'big'), sk)
        out[i:i+8] = c.to_bytes(8, 'big')
    return bytes(out)

def des_decrypt(data, key, pad=True):
    sk = key_schedule(int.from_bytes(key, 'big'))[::-1]
    out = bytearray(len(data))
    for i in range(0, len(data), 8):
        p = des_block(int.from_bytes(data[i:i+8], 'big'), sk)
        out[i:i+8] = p.to_bytes(8, 'big')
    return bytes(out[:-out[-1]]) if pad else bytes(out)

## Test avec l'exemple classique

Exemple de référence (J. Orlin Grabbe, « The DES Algorithm Illustrated ») :
clé `133457799BBCDFF1`, clair `0123456789ABCDEF` → chiffré attendu `85E813540F0AB405`.

On affiche aussi quelques valeurs intermédiaires pour illustrer chaque étape.

In [7]:
key = 0x133457799BBCDFF1
pt  = 0x0123456789ABCDEF
sk  = key_schedule(key)

def bits(x, n, g=8):
    s = format(x, f'0{n}b')
    return ' '.join(s[i:i+g] for i in range(0, n, g))

print("K+ (PC-1)   :", bits(permute(key, PC1, 64), 56, 7))
print("K1 (PC-2)   :", bits(sk[0], 48, 6))
ip = permute(pt, IP, 64)
print("IP(M)       :", bits(ip, 64))
L0, R0 = ip >> 32, ip & 0xFFFFFFFF
print("E(R0)       :", bits(permute(R0, E, 32), 48, 6))
print("E(R0)^K1    :", bits(permute(R0, E, 32) ^ sk[0], 48, 6))
print("S-box       :", bits(sbox_sub(permute(R0, E, 32) ^ sk[0]), 32, 4))
print("f(R0,K1)    :", bits(f_simple(R0, sk[0]), 32, 4))
print()
ct = des_block(pt, sk)
print("chiffré (version rapide)  :", format(ct, '016X'))
print("chiffré (version simple)  :", format(des_block_simple(pt, sk), '016X'))
print("déchiffré                 :", format(des_block(ct, sk[::-1]), '016X'))

K+ (PC-1)   : 1111000 0110011 0010101 0101111 0101010 1011001 1001111 0001111
K1 (PC-2)   : 000110 110000 001011 101111 111111 000111 000001 110010
IP(M)       : 11001100 00000000 11001100 11111111 11110000 10101010 11110000 10101010
E(R0)       : 011110 100001 010101 010101 011110 100001 010101 010101
E(R0)^K1    : 011000 010001 011110 111010 100001 100110 010100 100111
S-box       : 0101 1100 1000 0010 1011 0101 1001 0111
f(R0,K1)    : 0010 0011 0100 1010 1010 1001 1011 1011

chiffré (version rapide)  : 85E813540F0AB405
chiffré (version simple)  : 85E813540F0AB405
déchiffré                 : 0123456789ABCDEF


On retrouve bien `85E813540F0AB405`, avec les deux versions (simple et optimisée), et le déchiffrement redonne le clair.

Test rapide sur un message texte :

In [8]:
k = b"INSALYON"
m = b"Data Encryption Standard - TP 5IF"
c = des_encrypt(m, k)
print(c.hex())
print(des_decrypt(c, k))

226dfab1f7a734470847d2f299d2879f04d5aa1fed53b03e44c642e93128e87441282b5087761b61
b'Data Encryption Standard - TP 5IF'
